In [ ]:
import itertools
from enum import IntEnum

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation
from tqdm.notebook import tqdm

import gutpython

In [ ]:
%matplotlib inline

# Setup

In [ ]:
model = gutpython.GutPython(
    # parameters
    max_stuck_chance=50,
    low_stuck_bound=2,
    unstuck_chance=10,
    mid_stuck_conc=10.0,
    seed_chance=5.0,
    seed_percent=5.0,
    absorption=0.0,
    reserve_fraction=0.0,
    bifido_lactate_production=0.005,
    flow_dist=0.28,
    bifido_doub=330,
    desulfo_doub=330,
    bacteroid_doub=330,
    clost_doub=330,
    # initialization constants
    init_num_bifidos=23562,
    init_num_bacteroids=5490,
    init_num_closts=921,
    init_num_desulfos=70,
    # controls
    tick_in_flow=480,
    in_conc_bacteroids=0,
    in_conc_bifidos=0,
    in_conc_closts=0,
    in_conc_desulfos=0,
    cs_inflow=lambda t: 0.1 if t < 300 or t > 400 else 1.0,  # Chondroitin sulfate
    fo_inflow=lambda t: 25.0 if t < 400 or t > 500 else 50.0,  # fructooligosaccharide
    glucose_inflow=lambda t: 30.0 if t < 500 or t > 600 else 60.0,
    inulin_inflow=lambda t: 10.0 if t < 600 or t > 700 else 100.0,
    lactose_inflow=lambda t: 15.0 if t < 700 or t > 800 else 30.0,
    lactate_inflow=lambda t: 0.0 if t < 800 or t > 900 else 1.0,
)
model.setup()

In [ ]:
# glucose_max = np.max(model.glucose + model.glucose_reserve)
# fo_max = np.max(model.fo + model.fo_reserve)
# lactose_max = np.max(model.lactose + model.lactose_reserve)
# lactate_max = np.max(model.lactate + model.lactate_reserve)
# inulin_max = np.max(model.inulin + model.inulin_reserve)
# for _ in range(1000):
#     model.go()
#     glucose_max = max(glucose_max, np.max(model.glucose + model.glucose_reserve))
#     fo_max = max(fo_max, np.max(model.fo + model.fo_reserve))
#     lactose_max = max(lactose_max, np.max(model.lactose + model.lactose_reserve))
#     lactate_max = max(lactate_max, np.max(model.lactate + model.lactate_reserve))
#     inulin_max = max(inulin_max, np.max(model.inulin + model.inulin_reserve))

# glucose_max, fo_max, lactose_max, lactate_max, inulin_max

observed_field_maxes = {
    "glucose": 108.0,
    "fo": 90.0,
    "lactose": 54,
    "lactate": 180.0,
    "inulin": 36.0,
}

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(16, 6))

model.plot_agents(axs[0, 0])

for idx, field in enumerate(
    [
        "glucose",
        "fo",
        "lactose",
        "lactate",
        "inulin",
    ]
):
    r, c = divmod(idx + 1, 2)
    model.plot_field(
        axs[r, c], field_name=field, field_max=observed_field_maxes[field] * 1.001
    )
    axs[r, c].set_title(field)

for r, c in itertools.product(range(3), range(2)):
    axs[r, c].set_aspect(20)

In [ ]:
num_steps = 2000


class Task(IntEnum):
    html_anim = 0
    save_mpeg = 1
    make_hdf5 = 2


task = Task.save_mpeg

pbar = tqdm(total=num_steps, display=False)

# Run

In [ ]:
if task in {Task.html_anim, Task.save_mpeg}:

    mean_bacteroid_energy = [model.mean_bacteroid_energy]
    mean_bifido_energy = [model.mean_bifido_energy]
    mean_clost_energy = [model.mean_clost_energy]
    mean_desulfo_energy = [model.mean_desulfo_energy]

    bacteroid_count = []
    bifido_count = []
    clost_count = []
    desulfo_count = []

    excreted_bacteroids = []
    excreted_bifidos = []
    excreted_closts = []
    excreted_desulfos = []

    excreted_inulin = []
    excreted_fo = []
    excreted_lactose = []
    excreted_lactate = []
    excreted_glucose = []
    excreted_cs = []

    absorbed_inulin = []
    absorbed_fo = []
    absorbed_lactose = []
    absorbed_lactate = []
    absorbed_glucose = []
    absorbed_cs = []

    fig, axs = plt.subplots(3, 2, figsize=(10, 4), layout="constrained")

    for r, c in itertools.product(range(3), range(2)):
        axs[r, c].set_aspect(20)

    # noinspection PyUnusedLocal
    def animate(i):
        model.go()
        pbar.update()

        mean_bacteroid_energy.append(model.mean_bacteroid_energy)
        mean_bifido_energy.append(model.mean_bifido_energy)
        mean_clost_energy.append(model.mean_clost_energy)
        mean_desulfo_energy.append(model.mean_desulfo_energy)

        bacteroid_count.append(model.num_bacteroids)
        bifido_count.append(model.num_bifidos)
        clost_count.append(model.num_closts)
        desulfo_count.append(model.num_desulfos)

        excreted_bacteroids.append(model.excreted_bacteroids)
        excreted_bifidos.append(model.excreted_bifidos)
        excreted_closts.append(model.excreted_closts)
        excreted_desulfos.append(model.excreted_desulfos)

        excreted_inulin.append(model.excreted_inulin)
        excreted_fo.append(model.excreted_fo)
        excreted_lactose.append(model.excreted_lactose)
        excreted_lactate.append(model.excreted_lactate)
        excreted_glucose.append(model.excreted_glucose)
        excreted_cs.append(model.excreted_cs)

        absorbed_inulin.append(model.absorbed_inulin)
        absorbed_fo.append(model.absorbed_fo)
        absorbed_lactose.append(model.absorbed_lactose)
        absorbed_lactate.append(model.absorbed_lactate)
        absorbed_glucose.append(model.absorbed_glucose)
        absorbed_cs.append(model.absorbed_cs)

        model.plot_agents(axs[0, 0])
        axs[0, 0].set_title(f"Agents; t={model.ticks}")

        for idx, field in enumerate(
            [
                "glucose",
                "fo",
                "lactose",
                "lactate",
                "inulin",
            ]
        ):
            r, c = divmod(idx + 1, 2)
            model.plot_field(
                axs[r, c],
                field_name=field,
                field_max=observed_field_maxes[field] * 1.001,
            )
            axs[r, c].set_title(field)

        for r, c in itertools.product(range(3), range(2)):
            axs[r, c].set_aspect(20)
            axs[r, c].get_yaxis().set_visible(False)


    pbar.update()
    # noinspection PyTypeChecker
    anim = FuncAnimation(
        fig,
        animate,
        frames=num_steps,
        repeat=False,
        interval=100,  # half the default
    )

In [ ]:
display(pbar.container)

if task == Task.html_anim:
    # noinspection PyUnboundLocalVariable
    html_anim = HTML(anim.to_html5_video())
    display(html_anim)
elif task == Task.save_mpeg:
    # noinspection PyUnboundLocalVariable
    anim.save("model-run.mpg")

# Plots

In [ ]:
def smooth(a, n=20):
    a = np.array(a)
    weights = np.array([np.exp(-0.5 * np.abs(n) ** 0.5) for n in range(-n, n + 1)])
    return sum(
        [weight * np.roll(a, n) for weight, n in zip(weights, range(-n, n + 1))]
    ) / np.sum(weights)

In [ ]:
# noinspection PyUnboundLocalVariable
plt.plot(mean_bacteroid_energy)
# noinspection PyUnboundLocalVariable
plt.plot(mean_bifido_energy)
# noinspection PyUnboundLocalVariable
plt.plot(mean_clost_energy)
# noinspection PyUnboundLocalVariable
plt.plot(mean_desulfo_energy)

In [ ]:
# noinspection PyUnboundLocalVariable
plt.plot(bacteroid_count, color="grey")
# noinspection PyUnboundLocalVariable
plt.plot(bifido_count, color="blue")
# noinspection PyUnboundLocalVariable
plt.plot(clost_count, color="red")
# noinspection PyUnboundLocalVariable
plt.plot(desulfo_count, color="green")

In [ ]:
plt.plot(desulfo_count, color="green")

In [ ]:
# noinspection PyUnboundLocalVariable
plt.plot(excreted_bacteroids, label="bacteroids")
# noinspection PyUnboundLocalVariable
plt.plot(excreted_bifidos, label="bifidobacteria")
# noinspection PyUnboundLocalVariable
plt.plot(excreted_closts, label="clostridia")
# noinspection PyUnboundLocalVariable
plt.plot(excreted_desulfos, label="desulfovibro")
plt.legend()

In [ ]:
# noinspection PyUnboundLocalVariable
plt.plot(smooth(excreted_bacteroids), label="bacteroids")
# noinspection PyUnboundLocalVariable
plt.plot(smooth(excreted_bifidos), label="bifidobacteria")
# noinspection PyUnboundLocalVariable
plt.plot(smooth(excreted_closts), label="clostridia")
# noinspection PyUnboundLocalVariable
plt.plot(smooth(excreted_desulfos), label="desulfovibro")
plt.legend()

In [ ]:
plt.plot(excreted_inulin, label="inulin")
plt.plot(excreted_fo, label="fo")
plt.plot(excreted_lactose, label="lactose")
plt.plot(excreted_lactate, label="lactate")
plt.plot(excreted_glucose, label="glucose")
plt.plot(excreted_cs, label="cs")
plt.legend()

In [ ]:
plt.plot(excreted_inulin, label="inulin")
plt.plot(excreted_fo, label="fo")
plt.plot(excreted_lactose, label="lactose")
# plt.plot(excreted_lactate, label='lactate')
plt.plot(excreted_glucose, label="glucose")
plt.plot(excreted_cs, label="cs")
plt.legend()

In [ ]:
plt.plot(absorbed_inulin)
plt.plot(absorbed_fo)
plt.plot(absorbed_lactose)
plt.plot(absorbed_lactate)
plt.plot(absorbed_glucose)
plt.plot(absorbed_cs)

In [ ]:
plt.plot(np.array(bacteroid_count) / (1 + np.array(excreted_bacteroids)), color="grey")